<a href="https://colab.research.google.com/github/juanjosedelgado-coder/Talleres/blob/main/Taller_Apriori_Cuaspud_Vargas_Delgado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/4_Aprendizaje_no_supervisado/2_Taller_Apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# **Taller: Análisis de Patrones de Consumo Internacional con Apriori**

**IMPORTANTE**: Guarda una copia de este notebook en tu Google Drive o computador.

**Taller en grupos de 3**

**Nombres estudiantes:**

- Natalia Vargas
- Juan Jose Delgado

**Forma de entrega:**

- Nombrar el archivo de la siguiente forma:“Taller_Apriori_apellidos.ipynb”.
- Suba el Jupyter Notebook a su cuenta en Github y envíe el link en el siguiente Forms: https://forms.cloud.microsoft/r/qERdEpXpmx.

**IMPORTANTE:** No se recibirán talleres en Google Colab, el notebook debe estar subido en Github.

**Plazo de entrega:**

21 de abril de 2026, máximo a las 11:59 p.m. Tenga en cuenta que luego de esa hora el formulario en forms se cierra. El Jupupyter Notebook también debe quedar subido en Github antes de esa hora.

**Instrucciones Generales:**

Completa el código en las celdas marcadas con `### TU CÓDIGO AQUÍ ###`. Puedes añadir más celdas si lo requieres.

**Caso de Estudio: Consultoría para Global Retail Inc.**

**Contexto:** Una firma multinacional de e-commerce, "Global Retail Inc.", te ha contratado como consultor de datos. La empresa opera en múltiples países y ha notado que sus ventas y la efectividad de sus campañas de marketing varían significativamente entre regiones. Su hipótesis es que los patrones de compra y las asociaciones de productos son diferentes en cada mercado.

**Tu Misión:** Analizar el historial de transacciones de la empresa para descubrir y comparar las reglas de asociación de productos para dos de sus mercados más importantes en Latinoamérica: México y Colombia. Tu objetivo final es entregar recomendaciones de negocio accionables (ej. estrategias de cross-selling, promociones personalizadas) basadas en los patrones de consumo que descubras en cada país.

**Dataset:** Encuentra mayor información en el archivo "diccionario_alimentos_retail_top30.xlsx".

## Ejercicio 1: Configuración Inicial, Carga y Exploración de Datos

1.1 Importa las librerías necesarias

In [1]:
### TU CÓDIGO AQUÍ ###
import os
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Configuraciones de visualización
pd.options.display.max_columns = None
pd.options.display.float_format = '{:,.2f}'.format

1.2 Carga el dataset "alimentos_retail_top30.csv" que se encuentra en el repositorio del curso, carpeta "datasets". El dataframe debe llamarse "df".

In [3]:
### TU CÓDIGO AQUÍ ###
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
path = "/content/drive/MyDrive/IA Economia"
os.chdir(path)

In [5]:
df = pd.read_csv("alimentos_retail_top30.csv")

In [6]:
# Debe ser (6899, 8)
print("Dimensiones del DataFrame:")
print(df.shape)

Dimensiones del DataFrame:
(6899, 8)


In [7]:
print("\nInformación general del DataFrame:")
df.info()


Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6899 entries, 0 to 6898
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    6899 non-null   object 
 1   StockCode    6899 non-null   int64  
 2   Description  6899 non-null   object 
 3   Quantity     6899 non-null   int64  
 4   InvoiceDate  6899 non-null   object 
 5   UnitPrice    6899 non-null   float64
 6   CustomerID   6879 non-null   float64
 7   Country      6899 non-null   object 
dtypes: float64(2), int64(2), object(4)
memory usage: 431.3+ KB


1.3 Revisa si hay valores nulos en alguna columna y cuántos son

In [8]:
nulos = df.isnull().sum()
print(nulos)

InvoiceNo       0
StockCode       0
Description     0
Quantity        0
InvoiceDate     0
UnitPrice       0
CustomerID     20
Country         0
dtype: int64


1.4 Genera las estadísticas descriptivas de las variables numéricas

In [9]:
df.describe()

,StockCode,Quantity,UnitPrice,CustomerID
count,"6,899.00","6,899.00","6,899.00","6,879.00"
mean,"55,544.94",3.00,3.42,"15,024.12"
std,"25,875.73",1.43,1.06,"1,732.95"
min,"26,907.00",-5.00,1.65,"12,000.00"
25%,"31,048.00",2.00,2.36,"13,524.00"
50%,"42,889.00",3.00,3.39,"15,041.00"
75%,"87,297.00",4.00,4.44,"16,530.50"
max,"95,931.00",5.00,4.90,"17,999.00"


1.5 Observando las salidas del ejercicio anterior, ¿qué problemas potenciales identificas en las columnas CustomerID y Quantity?

**R/** En la columna CustomerID se observan valores faltantes, ya que el número de registros es menor al total del dataset, lo que indica la presencia de datos nulos. Esto puede afectar análisis posteriores relacionados con clientes.

En la columna Quantity, se identifican valores negativos, lo cual no es consistente con cantidades de compra. Esto sugiere la presencia de devoluciones o errores en los datos, lo cual debe ser tratado antes de realizar análisis.

## Ejercicio 2: Limpieza y Preprocesamiento de Datos

Los datos del mundo real rara vez son perfectos. Antes de cualquier análisis, debemos "sanear" nuestro dataset. Completa el código en cada paso según las instrucciones.

Crea un nuevo dataframe llamado "df_limpio" para los siguientes puntos.

2.1 **Manejo de Valores Nulos**: Las transacciones sin un CustomerID no son útiles para nosotros, ya que no podemos agrupar las compras de un cliente específico.

In [10]:
# TAREA: Elimina todas las filas donde 'CustomerID' es nulo.
df_limpio =  df.dropna(subset=['CustomerID'])

In [11]:
# El tipo de dato de CustomerID debe ser entero
df_limpio['CustomerID'] = df_limpio['CustomerID'].astype(int)

2.2 **Limpieza de Descripciones de Productos** Las descripciones pueden tener espacios en blanco al inicio o al final que podrían hacer que un mismo producto se cuente como dos diferentes.

In [12]:
# TAREA: # Verifica cuántas descripciones únicas hay.
df_limpio['Description'].nunique()

25

In [13]:
# TAREA: Limpia la columna 'Description' eliminando espacios extra al inicio y al final.
df_limpio['Description'] = df_limpio['Description'].str.strip()

In [14]:
# TAREA: Verifica cuántas descripciones únicas quedaron después de la limpieza.
df_limpio['Description'].nunique()

20

2.3 **Filtrado de Transacciones Anómalas**: Las facturas (InvoiceNo) que empiezan con 'C' indican una cancelación. Estas no son compras reales y deben ser eliminadas. Del mismo modo, las cantidades (Quantity) negativas representan devoluciones.

In [15]:
# TAREA: Elimina las filas que correspondan a cancelaciones.
df_limpio = df_limpio[~df_limpio['InvoiceNo'].str.startswith('C')]

In [16]:
# TAREA: Elimina las filas con cantidades negativas.
df_limpio = df_limpio[df_limpio['Quantity'] >= 0]

In [18]:
# Verifiquemos las dimensiones del DataFrame después de la limpieza. Debe ser (6864, 8)
df_limpio.shape

(6864, 8)

In [17]:
df_limpio.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536000,94537,HARINA DE MAÍZ,5,2023-01-07 01:09:00,2.76,17452,Colombia
1,536000,87297,QUESO MUZZARELLA,2,2023-01-07 01:09:00,4.69,17779,Colombia
2,536001,94537,HARINA DE MAÍZ,4,2023-01-07 11:51:00,2.76,14933,Colombia
3,536001,87297,QUESO MUZZARELLA,3,2023-01-07 11:51:00,4.69,14957,Colombia
4,536002,26907,CAFÉ,4,2023-01-02 01:54:00,2.36,15202,Colombia


## Ejercicio 3: Análisis Comparativo por País

Ahora que los datos están limpios, vamos a segmentarlos y a aplicar el algoritmo Apriori para encontrar los patrones de compra en México y Colombia.

**Preparación de la Cesta de Mercado (Función)**

La siguiente función toma un dataframe, lo agrupa por factura y descripción, y lo transforma en el formato de matriz binaria que necesita el algoritmo Apriori. Estudia esta función, no necesitas modificarla.

In [18]:
def preparar_cesta(dataframe, pais):
    """Filtra por país y prepara la matriz de transacciones."""

    # Filtrar por el país de interés
    df_pais = dataframe[dataframe['Country'] == pais]

    # Crear la cesta: agrupar productos por factura
    cesta = (df_pais.groupby(['InvoiceNo', 'Description'])['Quantity']
             .sum().unstack().reset_index().fillna(0)
             .set_index('InvoiceNo'))

    # Convertir todas las cantidades positivas a 1 y todo lo demás a 0
    cesta_encoded = (cesta > 0).astype(int)

    return cesta_encoded

3.1 Análisis para México

In [20]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de México. Almacena el resultado en la variable cesta_mx.
cesta_mx = preparar_cesta(df_limpio, 'México')
cesta_mx.head()

Description,AGUACATE,CEBOLLA,CHILE JALAPEÑO,CILANTRO,FRIJOL NEGRO,LIMÓN,QUESO FRESCO,TOMATE,TORTILLAS DE MAÍZ,TOTOPOS
InvoiceNo,,,,,,,,,,
537000,0,0,0,0,1,0,0,0,1,0
537001,0,1,1,1,0,0,0,1,0,0
537002,0,0,0,0,1,0,0,1,1,0
537003,0,1,1,1,0,0,0,1,0,0
537004,0,0,0,1,0,0,1,1,1,0


In [21]:
# TAREA: Aplica el algoritmo apriori para encontrar itemsets con un soporte mínimo de 2%.
# Almacena el resultado en la variable frequent_itemsets_mx.
# Muestra los 10 itemsets con el soporte más alto.
frequent_itemsets_mx = apriori(cesta_mx, min_support=0.02, use_colnames=True)
frequent_itemsets_mx.sort_values(by = 'support', ascending=False).head(10)

,support,itemsets
2,0.42,(CHILE JALAPEÑO)
7,0.41,(TOMATE)
3,0.41,(CILANTRO)
1,0.41,(CEBOLLA)
8,0.38,(TORTILLAS DE MAÍZ)
4,0.36,(FRIJOL NEGRO)
0,0.35,(AGUACATE)
5,0.35,(LIMÓN)
9,0.33,(TOTOPOS)
31,0.33,"(CHILE JALAPEÑO, TOMATE)"


In [22]:
# TAREA: Genera las reglas de asociación. Queremos reglas con un Lift mayor a 2. Almacena el resultado en la variable rules_mx.
rules_mx = association_rules(frequent_itemsets_mx, metric='lift', min_threshold=2)

In [23]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
rules_mx.sort_values(by=['lift', 'confidence'], ascending=[False, False]).head(10)[['antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift']]

,antecedents,consequents,antecedent support,consequent support,confidence,lift
88,"(CEBOLLA, CHILE JALAPEÑO)","(TOMATE, CILANTRO)",0.32,0.32,0.92,2.90
93,"(TOMATE, CILANTRO)","(CEBOLLA, CHILE JALAPEÑO)",0.32,0.32,0.93,2.90
90,"(CEBOLLA, CILANTRO)","(CHILE JALAPEÑO, TOMATE)",0.31,0.33,0.95,2.88
91,"(CHILE JALAPEÑO, TOMATE)","(CEBOLLA, CILANTRO)",0.33,0.31,0.90,2.88
12,"(LIMÓN, AGUACATE)",(TOTOPOS),0.27,0.33,0.93,2.82
13,(TOTOPOS),"(LIMÓN, AGUACATE)",0.33,0.27,0.75,2.82
92,"(CHILE JALAPEÑO, CILANTRO)","(CEBOLLA, TOMATE)",0.32,0.33,0.92,2.81
89,"(CEBOLLA, TOMATE)","(CHILE JALAPEÑO, CILANTRO)",0.33,0.32,0.91,2.81
74,"(LIMÓN, AGUACATE, TOMATE)",(TOTOPOS),0.02,0.33,0.91,2.76
75,(TOTOPOS),"(LIMÓN, AGUACATE, TOMATE)",0.33,0.02,0.06,2.76


3.3 Observa las 3 reglas con el Lift más alto para México (1, 3 y 5). **Interprétalas:** ¿Qué te dicen estas asociaciones? ¿Qué tipo de productos son?

**R/**

Las tres reglas con mayor lift evidencian patrones claros de consumo conjunto en el mercado mexicano. La primera regla indica que quienes compran cebolla y chile jalapeño también adquieren cilantro y tomate, lo que sugiere una combinación típica de ingredientes utilizados en preparaciones tradicionales como salsas frescas. De manera similar, la tercera regla muestra que la compra de cilantro y cebolla está fuertemente asociada con la compra de tomate y chile jalapeño, reforzando la idea de un conjunto base de ingredientes complementarios en la cocina mexicana. Por otro lado, la quinta regla relaciona el aguacate y el limón con la compra de totopos, lo que refleja un patrón de consumo orientado a snacks, específicamente la preparación de guacamole acompañado de totopos. En conjunto, estas asociaciones indican que los productos no son sustitutos, sino complementarios, y responden a hábitos culturales de consumo.

3.4 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift

**R/**

En la primera regla, el soporte del antecedente (0.32) indica que el 32% de las transacciones incluyen cebolla y chile jalapeño, mientras que el soporte del consecuente (0.32) muestra que el mismo porcentaje incluye cilantro y tomate. La confianza de 0.92 implica que el 92% de los consumidores que compran cebolla y jalapeño también adquieren cilantro y tomate, y el lift de 2.90 evidencia que esta relación es casi tres veces más probable que ocurra por azar. En la tercera regla, el soporte del antecedente es 0.31 y el del consecuente es 0.33, lo que indica una frecuencia similar en las compras de estos productos. La confianza alcanza 0.95, lo que refleja una asociación aún más fuerte, mientras que el lift de 2.88 confirma una relación altamente significativa. Finalmente, en la quinta regla, el soporte del antecedente (0.27) indica que el 27% de las compras incluyen aguacate y limón, y el soporte del consecuente (0.33) que el 33% incluye totopos. La confianza de 0.93 muestra que la gran mayoría de quienes compran los dos primeros productos también compran el tercero, y el lift de 2.82 confirma una fuerte relación positiva entre ellos.

3.5 **Recomendación de Negocio:** Basado en estas reglas, ¿qué promoción o estrategia de venta específica podrías sugerir para el mercado mexicano?

**R/**

A partir de estas reglas, se pueden plantear estrategias de negocio enfocadas en aumentar las ventas mediante la complementariedad de productos. Una opción es diseñar combos o paquetes promocionales, como un kit para preparar salsa mexicana que incluya tomate, cebolla, cilantro y chile jalapeño, o un kit para guacamole con aguacate, limón y totopos. También es recomendable ubicar estos productos de manera estratégica dentro del punto de venta para incentivar la compra conjunta, colocando, por ejemplo, los totopos cerca del aguacate o agrupando los ingredientes de salsa en una misma sección. Adicionalmente, se pueden implementar promociones cruzadas, ofreciendo descuentos en productos complementarios al comprar ciertos artículos. Finalmente, es útil reforzar estas estrategias con campañas de marketing que destaquen la preparación de alimentos tradicionales, incentivando así el consumo conjunto y aumentando el valor promedio de compra.

3.6 Análisis para Colombia

In [24]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de Colombia. Almacena el resultado en la variable cesta_co.
cesta_co = preparar_cesta(df_limpio, 'Colombia')
cesta_co.head()

Description,ACEITE DE GIRASOL,ARROZ,AZÚCAR,CAFÉ,FRIJOL CARGAMANTO,HARINA DE MAÍZ,HUEVOS,LECHE,PAN TAJADO,QUESO MUZZARELLA
InvoiceNo,,,,,,,,,,
536000,0,0,0,0,0,1,0,0,0,1
536001,0,0,0,0,0,1,0,0,0,1
536002,0,0,1,1,0,0,0,1,0,0
536003,0,0,0,0,0,0,1,0,1,0
536004,1,1,0,0,1,0,0,0,0,0


In [25]:
# TAREA: Aplica el algoritmo apriori con un soporte mínimo del 2%.
# Almacena el resultado en la variable frequent_itemsets_co.
# Muestra los 10 itemsets con el soporte más alto.
frequent_itemsets_co = apriori(cesta_co, min_support=0.02, use_colnames=True)
frequent_itemsets_co.sort_values(by = 'support', ascending=False).head(10)

,support,itemsets
3,0.41,(CAFÉ)
4,0.41,(FRIJOL CARGAMANTO)
0,0.40,(ACEITE DE GIRASOL)
2,0.40,(AZÚCAR)
7,0.39,(LECHE)
1,0.38,(ARROZ)
9,0.35,(QUESO MUZZARELLA)
5,0.34,(HARINA DE MAÍZ)
37,0.32,"(CAFÉ, LECHE)"
31,0.32,"(AZÚCAR, LECHE)"


In [26]:
# TAREA: Genera las reglas de asociación con un Lift mayor a 2. Almacena el resultado en la variable rules_co.
rules_co = association_rules(frequent_itemsets_co, metric='lift', min_threshold=2)

In [27]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
rules_co.sort_values(by=['lift', 'confidence'], ascending=[False, False]).head(10)[['antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift']]

,antecedents,consequents,antecedent support,consequent support,confidence,lift
41,"(FRIJOL CARGAMANTO, AZÚCAR, CAFÉ)",(LECHE),0.05,0.39,0.96,2.44
44,(LECHE),"(FRIJOL CARGAMANTO, AZÚCAR, CAFÉ)",0.39,0.05,0.11,2.44
11,"(AZÚCAR, CAFÉ)",(LECHE),0.32,0.39,0.95,2.41
14,(LECHE),"(AZÚCAR, CAFÉ)",0.39,0.32,0.76,2.41
5,"(FRIJOL CARGAMANTO, ACEITE DE GIRASOL)",(ARROZ),0.30,0.38,0.91,2.41
8,(ARROZ),"(FRIJOL CARGAMANTO, ACEITE DE GIRASOL)",0.38,0.30,0.73,2.41
59,"(AZÚCAR, PAN TAJADO, CAFÉ)",(LECHE),0.03,0.39,0.94,2.39
66,(LECHE),"(AZÚCAR, PAN TAJADO, CAFÉ)",0.39,0.03,0.07,2.39
49,"(HUEVOS, AZÚCAR, CAFÉ)",(LECHE),0.04,0.39,0.92,2.36
56,(LECHE),"(HUEVOS, AZÚCAR, CAFÉ)",0.39,0.04,0.09,2.36


3.7 Observa las 3 reglas con el Lift más alto para Colombia (1, 3 y 5). **Interprétalas:** ¿Qué patrones de consumo específicos del mercado colombiano revelan estas reglas? ¿Son diferentes a las de México?

**R/** Las reglas con mayor lift para Colombia muestran patrones de consumo centrados en productos básicos de la canasta familiar y hábitos cotidianos. La primera regla indica que quienes compran azúcar, frijol cargamanto y café también adquieren leche, lo que sugiere un patrón de consumo asociado al desayuno o bebidas calientes acompañadas de alimentos tradicionales. La tercera regla refuerza esta idea, mostrando que la combinación de azúcar y café tiene una fuerte relación con la compra de leche, lo que evidencia el consumo típico de café con leche en los hogares colombianos. Por su parte, la quinta regla indica que quienes compran aceite de girasol y frijol cargamanto también compran arroz, lo que refleja la base de la alimentación diaria en Colombia, donde estos productos son fundamentales en la preparación de comidas caseras.

En comparación con México, los patrones son diferentes. Mientras en México predominan combinaciones asociadas a preparaciones específicas como salsas o snacks (guacamole), en Colombia se observan asociaciones más relacionadas con el consumo diario, especialmente desayunos y almuerzos tradicionales, lo que refleja una canasta más orientada a productos básicos y menos a preparaciones específicas.

3.8 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift

**R/** En la primera regla, el soporte del antecedente (0.05) indica que el 5% de las transacciones incluyen la combinación de azúcar, frijol cargamanto y café, mientras que el soporte del consecuente (0.39) muestra que el 39% de las compras incluyen leche. La confianza de 0.96 implica que el 96% de quienes compran estos tres productos también adquieren leche, y el lift de 2.44 indica que esta asociación es más de dos veces superior a lo esperado por azar, evidenciando una relación muy fuerte.

En la tercera regla, el soporte del antecedente (0.32) indica que el 32% de las compras incluyen azúcar y café, mientras que el soporte del consecuente (0.39) corresponde a la leche. La confianza de 0.95 muestra que el 95% de quienes compran azúcar y café también compran leche, y el lift de 2.41 confirma una asociación significativa entre estos productos, consistente con hábitos de consumo como el café con leche.

En la quinta regla, el soporte del antecedente (0.30) indica que el 30% de las transacciones incluyen aceite de girasol y frijol cargamanto, mientras que el soporte del consecuente (0.38) muestra la frecuencia de compra del arroz. La confianza de 0.91 indica que el 91% de quienes compran los dos primeros productos también adquieren arroz, y el lift de 2.41 señala una relación fuerte que refleja la complementariedad en la preparación de comidas tradicionales.

3.9 **Recomendación de Negocio:** ¿Qué campaña de marketing (diferente a la de México) podrías diseñar para los clientes colombianos?

**R/** A partir de estas reglas, se puede diseñar una estrategia de marketing enfocada en los hábitos cotidianos del consumidor colombiano, especialmente en torno al desayuno y la alimentación básica del hogar. Una campaña efectiva podría centrarse en la creación de “combos de desayuno”, incluyendo productos como café, azúcar y leche, promovidos como esenciales para iniciar el día. Asimismo, se podrían ofrecer paquetes de “mercado básico” que integren arroz, frijoles y aceite, resaltando su papel en la alimentación diaria.

A diferencia del caso mexicano, donde la estrategia se enfocaba en preparaciones específicas, en Colombia la campaña debería resaltar la practicidad, el ahorro y la tradición, apelando a la rutina diaria de los hogares. También se podrían implementar descuentos por la compra conjunta de estos productos y ubicarlos estratégicamente en tienda para facilitar su adquisición. Este enfoque permitiría aumentar el valor promedio de compra y fortalecer la fidelización del cliente al alinearse con sus hábitos de consumo reales.